# 00_01 - Tres en raya: humano vs agente

Cuadernillo complementario a `00_intro.ipynb`. Cargamos `agente.pickle` (funcion de valor entrenada) y jugamos contra el modelo.

In [1]:
import numpy as np
import pickle

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
with open('/content/drive/MyDrive/Colab Notebooks/machine learning/07 aprendizaje por refuerzo/agente.pickle', 'rb') as handle:
    funcion_de_valor = pickle.load(handle)

len(funcion_de_valor)

5477

In [4]:
class Board:
    def __init__(self):
        self.state = np.zeros((3, 3))

    def valid_moves(self):
        return [(i, j) for j in range(3) for i in range(3) if self.state[i, j] == 0]

    def update(self, symbol, row, col):
        if self.state[row, col] != 0:
            raise ValueError('movimiento ilegal')
        self.state[row, col] = symbol

    def is_game_over(self):
        if (self.state.sum(axis=0) == 3).sum() >= 1 or (self.state.sum(axis=1) == 3).sum() >= 1:
            return 1
        if (self.state.sum(axis=0) == -3).sum() >= 1 or (self.state.sum(axis=1) == -3).sum() >= 1:
            return -1

        diag_sums = [
            sum(self.state[i, i] for i in range(3)),
            sum(self.state[i, 3 - i - 1] for i in range(3)),
        ]
        if diag_sums[0] == 3 or diag_sums[1] == 3:
            return 1
        if diag_sums[0] == -3 or diag_sums[1] == -3:
            return -1

        if len(self.valid_moves()) == 0:
            return 0

        return None


class AgentInferencia:
    def __init__(self, value_function, symbol=1):
        self.value_function = value_function
        self.symbol = symbol

    def move(self, board):
        valid_moves = board.valid_moves()
        max_value = -1000
        best_row, best_col = valid_moves[0]
        for row, col in valid_moves:
            next_board = board.state.copy()
            next_board[row, col] = self.symbol
            next_state = str(next_board.reshape(9))
            value = self.value_function.get(next_state, 0)
            if value >= max_value:
                max_value = value
                best_row, best_col = row, col
        return best_row, best_col

In [5]:
def dibujar_tablero(board):
    simbolos = {1: 'X', -1: 'O', 0: ' '}
    print('')
    for r in range(3):
        fila = []
        for c in range(3):
            if board.state[r, c] == 0:
                fila.append(str(r * 3 + c + 1))
            else:
                fila.append(simbolos[int(board.state[r, c])])
        print(' ' + ' | '.join(fila))
        if r < 2:
            print('---+---+---')
    print('')


def movimiento_humano(board):
    valid_numbers = {r * 3 + c + 1: (r, c) for r, c in board.valid_moves()}
    while True:
        entrada = input(f'Tu turno (elige {sorted(valid_numbers.keys())}): ').strip()
        if not entrada.isdigit():
            print('Entrada invalida. Escribe un numero.')
            continue
        numero = int(entrada)
        if numero not in valid_numbers:
            print('Esa casilla no esta disponible.')
            continue
        return valid_numbers[numero]


def jugar(agente_empieza=True):
    board = Board()
    agente = AgentInferencia(funcion_de_valor, symbol=1)
    simbolo_humano = -1

    print('Humano: O | Agente: X')
    dibujar_tablero(board)

    turno_agente = agente_empieza
    while board.is_game_over() is None:
        if turno_agente:
            row, col = agente.move(board)
            board.update(agente.symbol, row, col)
            print(f'Agente juega en casilla {row * 3 + col + 1}')
        else:
            row, col = movimiento_humano(board)
            board.update(simbolo_humano, row, col)

        dibujar_tablero(board)
        turno_agente = not turno_agente

    resultado = board.is_game_over()
    if resultado == 1:
        print('Gana el agente.')
    elif resultado == -1:
        print('Ganaste.')
    else:
        print('Empate.')

In [ ]:
# Ejecuta esta celda para jugar una partida.
# Cambia a False si quieres empezar tu.
jugar(agente_empieza=False)

Humano: O | Agente: X

 1 | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | 9

Tu turno (elige [1, 2, 3, 4, 5, 6, 7, 8, 9]): 1

 O | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | 9

Agente juega en casilla 5

 O | 2 | 3
---+---+---
 4 | X | 6
---+---+---
 7 | 8 | 9

Tu turno (elige [2, 3, 4, 6, 7, 8, 9]): 9

 O | 2 | 3
---+---+---
 4 | X | 6
---+---+---
 7 | 8 | O

Agente juega en casilla 8

 O | 2 | 3
---+---+---
 4 | X | 6
---+---+---
 7 | X | O

Tu turno (elige [2, 3, 4, 6, 7]): 2

 O | O | 3
---+---+---
 4 | X | 6
---+---+---
 7 | X | O

Agente juega en casilla 3

 O | O | X
---+---+---
 4 | X | 6
---+---+---
 7 | X | O

Tu turno (elige [4, 6, 7]): 4

 O | O | X
---+---+---
 O | X | 6
---+---+---
 7 | X | O

Agente juega en casilla 6

 O | O | X
---+---+---
 O | X | X
---+---+---
 7 | X | O

